# Full Analysis Pipeline

End-to-end network monitoring pipeline: load data from all three protocols, run exploratory analysis, detect anomalies, and diagnose root causes.

In [ ]:
import sys
sys.path.insert(0, '..')

from inventor_analysis.loaders import load_ping_data, load_dns_data, load_traceroute_data
from inventor_analysis.ping_analysis import (
    compute_host_statistics,
    detect_anomalies_zscore,
    compute_reliability_scores,
)
from inventor_analysis.dns_analysis import (
    compute_resolver_statistics,
    rank_resolvers,
)
from inventor_analysis.traceroute_analysis import (
    compute_hop_statistics,
    compute_gateway_performance,
    build_hop_baselines,
)
from inventor_analysis.anomaly_detection import (
    AdaptiveBaselineDetector,
    LatencyShiftDetector,
    JitterDiagnosisMatrix,
)

## 1. Load All Data Sources

In [ ]:
data_dir = "../sample_data"

ping_df = load_ping_data(data_dir, max_files=5)
dns_df = load_dns_data(data_dir, max_files=5)
trace_summary, trace_hops = load_traceroute_data(data_dir, max_files=5)

print(f"Ping: {len(ping_df):,} records")
print(f"DNS:  {len(dns_df):,} records")
print(f"Traceroute summaries: {len(trace_summary):,}")
print(f"Traceroute hops: {len(trace_hops):,}")

## 2. Exploratory Analysis

### Ping — Host Statistics

In [ ]:
if not ping_df.empty:
    host_stats = compute_host_statistics(ping_df)
    print(f"{host_stats.shape[0]} hosts analyzed")
    print(f"Overall avg RTT: {ping_df['rtt_avg'].mean():.2f} ms")
    print(f"Overall avg jitter: {ping_df['jitter'].mean():.2f} ms")
    display(host_stats)

### Ping — Reliability

In [ ]:
if not ping_df.empty:
    reliability = compute_reliability_scores(ping_df)
    display(reliability)

### DNS — Resolver Statistics

In [ ]:
if not dns_df.empty:
    resolver_stats = compute_resolver_statistics(dns_df)
    display(resolver_stats)

### Traceroute — Hop Statistics

In [ ]:
if not trace_hops.empty:
    hop_stats = compute_hop_statistics(trace_hops)
    print(f"{hop_stats.shape[0]} hop positions analyzed")
    display(hop_stats)

## 3. Anomaly Detection

In [ ]:
if not ping_df.empty:
    split = int(len(ping_df) * 0.7)
    train = ping_df.iloc[:split]
    test = ping_df.iloc[split:]

    detector = AdaptiveBaselineDetector()
    detector.train(train)
    predictions, scores, reasons = detector.predict(test)
    anomaly_count = predictions.sum()
    print(f"Adaptive Baseline: {anomaly_count} anomalies in {len(test)} test samples")

    df_zscore = detect_anomalies_zscore(ping_df, threshold=3.0)
    zscore_count = df_zscore['is_anomaly'].sum()
    print(f"Z-Score (>3 std): {zscore_count} anomalies in {len(ping_df)} total samples")

## 4. Latency Shift Detection (CUSUM)

In [ ]:
if not ping_df.empty:
    shift_detector = LatencyShiftDetector(sensitivity=5.0)
    shifts = shift_detector.detect(ping_df)
    print(f"Latency shifts detected: {len(shifts)}")

    if not shifts.empty:
        print(f"Avg shift magnitude: {shifts['rtt_change_pct'].mean():+.1f}%")
        display(shifts.head())

## 5. Root Cause Diagnosis

In [ ]:
if not ping_df.empty:
    diagnoser = JitterDiagnosisMatrix(percentile=0.95)
    diagnoser.train(train)
    diagnoses = diagnoser.diagnose(test)

    if not diagnoses.empty:
        print(f"Diagnosis results ({len(diagnoses)} non-normal events):")
        display(diagnoses['diagnosis'].value_counts())
    else:
        print("All measurements normal")

## Summary

In [ ]:
if not ping_df.empty:
    print(f"Monitored {ping_df['target_host'].nunique()} hosts over "
          f"{(ping_df['timestamp'].max() - ping_df['timestamp'].min()).days + 1} days")
    print(f"Average RTT: {ping_df['rtt_avg'].mean():.2f}ms, "
          f"Jitter: {ping_df['jitter'].mean():.2f}ms")
    if anomaly_count > 0:
        print(f"{anomaly_count} anomalies detected by adaptive baseline")
    if not shifts.empty:
        print(f"{len(shifts)} latency shifts detected")